# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [1]:
# Import necessary libraries

import random

import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
# Import your chosen baseline model
# Example: from sklearn.linear_model import LogisticRegression
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from datasets import load_dataset


RANDOM_STATE = 42


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(RANDOM_STATE)


In [2]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")  # NVIDIA GPU
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")   # Apple Silicon (M1/M2/M3)
else:
    DEVICE = torch.device("cpu")   # Fallback

print(f"Using Device: {DEVICE}")

Using Device: mps


## Model Choice

[Explain why you've chosen a particular model as the baseline. This could be a simple statistical model or a basic machine learning model. Justify your choice.]


We chose a convolutional neural network (CNN) as the baseline model for our aesthetic emotions map project. CNNs are well-suited for image classification tasks due to their ability to capture spatial hierarchies in images. They can learn to recognize patterns and features in the images that are relevant for predicting the associated emotions. Additionally, CNNs have been widely used and have shown strong performance in various image-related tasks, making them a reasonable starting point for our project.

## Feature Selection

[Indicate which features from the dataset you will be using for the baseline model, and justify your selection.]

For the baseline model, we will be using the raw pixel values of the images as features. This is a common approach for image classification tasks, as it allows the model to learn directly from the visual data without any manual feature engineering. By using the raw pixel values, we can leverage the CNN's ability to automatically extract relevant features from the images during training.

In [3]:
# Loading the dataset using Hugging Face's datasets library
dataset = load_dataset("bjoern-doege/aesthetic-emotions-map")

Resolving data files:   0%|          | 0/4721 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1025 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1089 [00:00<?, ?it/s]

In [4]:
NORMALIZE_MEAN=[0.5111283659934998, 0.48830345273017883, 0.46479079127311707]
NORMALIZE_STD=[0.3433663249015808, 0.3207928538322449, 0.32255250215530396]

In [5]:
BASE_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

In [6]:
# Converting labels to ids
# PyTorch's ImageFolder would do this automatically, but we are using Hugging Face's datasets library
label_names = sorted(dataset["train"].unique("label"))
label_to_id = {label: i for i, label in enumerate(label_names)}

In [7]:
# This function will be applied to each example in the dataset to preprocess the images and labels
# Again, this is necessary because we are using Hugging Face's datasets library instead of PyTorch's ImageFolder
def preprocess(examples):
    return {
        "image": [
            BASE_TRANSFORM(image.convert("RGB"))
            for image in examples["image"]
        ],
        "label": torch.tensor(
            [label_to_id[label] for label in examples["label"]],
            dtype=torch.long,
        ),
    }

In [8]:
# Apply the preprocessing function to the splitted datasets
train_dataset = dataset["train"].with_transform(preprocess)
test_dataset = dataset["test"].with_transform(preprocess)

In [9]:
train_generator = torch.Generator()
train_generator.manual_seed(RANDOM_STATE)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    generator=train_generator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
)

## Implementation

[Implement your baseline model here.]

The baseline CNN architecture consists of the following layers assuming an input image size of 224x224 pixels:
1. **Convolutional Layer 1**: 32 filters, kernel size of 3x3, ReLU activation
2. **Max Pooling Layer 1**: Pool size of 2x2
3. **Convolutional Layer 2**: 64 filters, kernel size of 3x3, ReLU activation
4. **Max Pooling Layer 2**: Pool size of 2x2
5. **Convolutional Layer 3**: 128 filters, kernel size of 3x3, ReLU activation
6. **Max Pooling Layer 3**: Pool size of 2x2
7. **Flatten Layer**: Flattens the output from the previous layer
8. **Fully Connected Layer 1**: 256 neurons, ReLU activation
9. **Dropout Layer**: Dropout rate of 0.5 to reduce overfitting
10. **Output Layer**: Number of neurons equal to the number of emotion classes

In [10]:
# Initialize and train the baseline model
# Example for a classification problem using Logistic Regression
# model = LogisticRegression()
# model.fit(X_train, y_train)

# Your implementation code here
class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Block 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Block 3
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # For 224x224 input images:
            # 224 -> 112 -> 56 -> 28
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [11]:
seed_everything(RANDOM_STATE)
model = BaselineCNN(num_classes=8).to(DEVICE)

In [12]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        # Forward pass
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Metrics
        running_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f}")

Epoch 1/10
Train loss: 1.8787, Train acc: 0.3138
Epoch 2/10
Train loss: 1.5465, Train acc: 0.4025
Epoch 3/10
Train loss: 1.3924, Train acc: 0.4752
Epoch 4/10
Train loss: 1.2340, Train acc: 0.5434
Epoch 5/10
Train loss: 1.0437, Train acc: 0.6089
Epoch 6/10
Train loss: 0.8480, Train acc: 0.6854
Epoch 7/10
Train loss: 0.6676, Train acc: 0.7619
Epoch 8/10
Train loss: 0.5609, Train acc: 0.8008
Epoch 9/10
Train loss: 0.4359, Train acc: 0.8481
Epoch 10/10
Train loss: 0.3616, Train acc: 0.8763


## Evaluation

[Clearly state what metrics you will use to evaluate the model's performance. These metrics will serve as a starting point for evaluating more complex models later on.]

We are using the following metrics to evaluate the performance of our baseline model:
1. **Accuracy**: The proportion of correctly classified instances among the total instances.
2. **Precision**: The proportion of true positive predictions among all positive predictions.
3. **Recall**: The proportion of true positive predictions among all actual positives.
4. **F1 Score**: The harmonic mean of precision and recall, providing a balance between the two metrics.
5. **Confusion Matrix**: A table that describes the performance of the classification model by showing the true positives, true negatives, false positives, and false negatives for each class.
6. **Classification Report**: A comprehensive report that includes precision, recall, F1 score, and support for each class, providing a detailed overview of the model's performance across all classes.

In [14]:
# Evaluate the baseline model
# Example for a classification problem
# y_pred = model.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)

# For a regression problem, you might use:
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here
model.eval()

test_loss = 0.0
y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        outputs = model(images)
        loss = loss_fn(outputs, labels)

        test_loss += loss.item()

        predicted = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

test_loss = test_loss / len(test_loader)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print(f"Test loss: {test_loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(label_names))),
    target_names=label_names
))

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

Test loss: 2.9650
Accuracy: 0.3998
Precision: 0.3883
Recall: 0.3998
F1-score: 0.3748

Classification report:
                       precision    recall  f1-score   support

  aesthetic_emptiness       0.34      0.30      0.32        80
energetic_playfulness       0.00      0.00      0.00        64
  intellectual_unease       0.38      0.72      0.50       240
   lighthearted_humor       0.59      0.44      0.51       144
          melancholic       0.31      0.23      0.26        96
            pure_calm       0.52      0.34      0.41       128
        serene_beauty       0.25      0.11      0.15       128
   sublime_activation       0.43      0.46      0.44       208

             accuracy                           0.40      1088
            macro avg       0.35      0.32      0.32      1088
         weighted avg       0.39      0.40      0.37      1088


Confusion matrix:
[[ 24   0  32   6  12   3   1   2]
 [  2   0  24  16   0   4   2  16]
 [  3   0 172   0   9   0  11  45]
 [ 11  1

In [15]:
torch.save(model.state_dict(), "model_seed_42.pth")